# VaultStaticSecret aggregation end-to-end test

## References
- [Support aggregation of multiple Vault paths into a single Kubernetes Secret](https://github.com/hashicorp/vault-secrets-operator/issues/1047)

## Start Kubernetes Cluster with kind (Kubernetes in Docker)

In [ ]:
!kind delete cluster --name vault-operator

In [ ]:
%%bash
kind create cluster --name vault-operator --config=<(cat <<EOF
kind: Cluster
apiVersion: kind.x-k8s.io/v1alpha4
nodes:
  - role: control-plane
    kubeadmConfigPatches:
      - |
        kind: InitConfiguration
        nodeRegistration:
          kubeletExtraArgs:
            node-labels: "ingress-ready=true"
    extraPortMappings:
      - containerPort: 80
        hostPort: 80
        protocol: TCP
EOF
)

In [ ]:
!kubectl apply -f https://kind.sigs.k8s.io/examples/ingress/deploy-ingress-nginx.yaml

In [ ]:
!sleep 10 && kubectl wait --namespace ingress-nginx \
  --for=condition=ready pod \
  --selector=app.kubernetes.io/component=controller \
  --timeout=90s

## Install Hashicorp Vault to Kubernetes

In [ ]:
!helm repo add hashicorp https://helm.releases.hashicorp.com
!helm repo update
!helm search repo hashicorp/vault

In [ ]:
%%bash
helm install vault hashicorp/vault -n vault --create-namespace \
  --values <(cat <<EOF
server:
  dev:
    enabled: true
    devRootToken: "root"
  logLevel: debug
ui:
  enabled: true
  serviceType: "LoadBalancer"
  externalPort: 8200

injector:
  enabled: "false"
)

In [ ]:
!kubectl wait --namespace vault \
  --for=condition=ready pod \
  --selector=app.kubernetes.io/name=vault \
  --timeout=30s

### Create an Ingress Route to make the HashiCorp Vault User Interface accessible.

In [ ]:
%%bash

kubectl apply -f - <<EOF
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: vaultingress
  namespace: vault
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /
spec:
  rules:
  - http:
      paths:
      - path: /
        pathType: Prefix
        backend:
          service:
            name: vault-ui
            port:
              number: 8200
EOF

In [ ]:
!curl http://localhost/ui/

## Configure Hashicorp Vault

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault auth enable -path demo-auth-mount kubernetes'

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault write auth/demo-auth-mount/config kubernetes_host="https://$KUBERNETES_PORT_443_TCP_ADDR:443"'

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault secrets enable -path=kvv2 kv-v2'

In [ ]:
%%bash
kubectl exec -i vault-0 -n vault -- vault policy write test-vault - <<EOF
path "kvv2/data/test-vault/*" {
   capabilities = ["read", "list"]
}
EOF

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault write auth/demo-auth-mount/role/role1 \
   bound_service_account_namespaces="vso-test" \
   bound_service_account_names=* \
   policies=test-vault \
   audience=vault \
   ttl=24h'

## Install the Vault Secrets Operator with inline-placeholder-support

Use this instead of the official chart above when testing `spec.data`. It builds the dev image from the `inline-placeholder-support` worktree (`linux/arm64`), loads it into the `vault-operator` Kind cluster, and upgrades the release from the local chart so the CRD includes `spec.data`.

In [ ]:
%%bash
WORKTREE=$(git rev-parse --show-toplevel)
IMG=hashicorp/vault-secrets-operator:inline-placeholder

cd "$WORKTREE"
docker build -t "$IMG" . --target=dev --load \
  --build-arg GOOS=linux \
  --build-arg GOARCH=arm64 \
  --build-arg GO_VERSION="$(cat .go-version)"

kind load docker-image --name vault-operator "$IMG"

helm upgrade --install vault-secrets-operator "$WORKTREE/chart" \
  -n vault-secrets-operator-system --create-namespace \
  --values <(cat <<EOF
defaultVaultConnection:
  enabled: true
  address: "http://vault.vault.svc.cluster.local:8200"
  skipTLSVerify: false
controller:
  manager:
    image:
      repository: hashicorp/vault-secrets-operator
      tag: inline-placeholder
      pullPolicy: IfNotPresent
    clientCache:
      persistenceModel: direct-encrypted
      storageEncryption:
        enabled: true
        mount: demo-auth-mount
        keyName: vso-client-cache
        transitMount: demo-transit
        kubernetes:
          role: auth-role-operator
          serviceAccount: vault-secrets-operator-controller-manager
          tokenAudiences: ["vault"]
EOF
)

In [ ]:
!kubectl rollout restart deployment -n vault-secrets-operator-system -l app.kubernetes.io/name=vault-secrets-operator && \
kubectl wait --namespace vault-secrets-operator-system \
  --for=condition=ready pod \
  --selector=app.kubernetes.io/name=vault-secrets-operator \
  --timeout=90s

## Create Test Namespace and Dependencies

In [ ]:
!kubectl create ns vso-test

In [ ]:
%%bash
kubectl create -f - <<EOF
apiVersion: v1
kind: ServiceAccount
metadata:
  namespace: vso-test
  name: demo-static-app
---
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultAuth
metadata:
  name: static-auth
  namespace: vso-test
spec:
  method: kubernetes
  mount: demo-auth-mount
  kubernetes:
    role: role1
    serviceAccount: demo-static-app
    audiences:
      - vault
EOF

## Test Standard VaultStaticSecret

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv put kvv2/test-vault/config username="static-user" password="static-password"'

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv get kvv2/test-vault/config'

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  name: vault-test-secret
  namespace: vso-test
spec:
  type: kv-v2

  # mount path
  mount: kvv2

  # path of the secret
  path: test-vault/config

  # dest k8s secret
  destination:
    name: vault-test-secret
    create: true

  # static secret refresh interval
  refreshAfter: 10s

  # Name of the CRD to authenticate to Vault
  vaultAuthRef: static-auth
EOF

In [ ]:
!kubectl get VaultStaticSecret -n vso-test vault-test-secret  -o yaml

In [ ]:
!kubectl get secret vault-test-secret -n vso-test -o go-template='{{range $k, $v := .data}}{{$k}}: {{$v | base64decode}}{{"\n"}}{{end}}'

### Rotate Secret Values

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv put kvv2/test-vault/config username="static-user-new" password="static-password-new"'

## Test VaultStaticSecret with Inline Placeholders

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv put kvv2/test-vault/infrastructure/acs api_token="1234567890abcdef" api_url="https://acs.it-rat.com"'
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv put kvv2/test-vault/infrastructure/gitlab token="test-token"'

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv get kvv2/test-vault/infrastructure/acs'
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv get kvv2/test-vault/infrastructure/gitlab'

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: secrets.hashicorp.com/v1beta1
kind: VaultStaticSecret
metadata:
  name: vault-test-secret-aggregated
  namespace: vso-test
spec:
  type: kv-v2

  # mount path
  mount: kvv2

  # path of the secret
  data:
    username: test-vault/config#username
    password: test-vault/config#password
    acs_api_token: test-vault/infrastructure/acs#api_token
    gitlab_api_token: test-vault/infrastructure/gitlab#token

  # dest k8s secret
  destination:
    name: vault-test-secret-aggregated
    create: true

  # static secret refresh interval
  refreshAfter: 10s

  # Name of the CRD to authenticate to Vault
  vaultAuthRef: static-auth
EOF

In [ ]:
!kubectl get VaultStaticSecret -n vso-test vault-test-secret-aggregated  -o yaml

In [ ]:
!kubectl get secret vault-test-secret-aggregated -n vso-test -o go-template='{{range $k, $v := .data}}{{$k}}: {{$v | base64decode}}{{"\n"}}{{end}}'

### Rotate Multiple Secret Values

In [ ]:
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv put kvv2/test-vault/infrastructure/acs api_token="1234567890abcdef-new" api_url="https://acs.it-rat.com"'
!kubectl exec -i vault-0 -n vault -- /bin/sh -c 'vault kv put kvv2/test-vault/infrastructure/gitlab token="test-token-new"'

In [ ]:
!kubectl get secret vault-test-secret-aggregated -n vso-test -o go-template='{{range $k, $v := .data}}{{$k}}: {{$v | base64decode}}{{"\n"}}{{end}}'